# RF-DETR tank detector: train on Kaggle, export for the browser

Plan: fine-tune RF-DETR Nano with the `rfdetr` package, check it in Python, then export **ONNX** (main browser route, onnxruntime-web on WebGPU) and **LiteRT** (`.tflite`).

**Before running:** Accelerator → GPU, Internet → On, add your dataset as an input.

Dataset format (COCO detection, Roboflow's "COCO" export already looks like this):
```
your-dataset/
  train/  images + _annotations.coco.json
  valid/  images + _annotations.coco.json
```

In [ ]:
!pip install -q --no-warn-conflicts rfdetr supervision onnx onnxruntime onnxconverter-common

## 1. Config
Run this cell again first if you ever restart the session; every section below only needs these values and the checkpoint on disk.

In [ ]:
import os, glob, shutil

# ---- EDIT THESE ----
DATASET_DIR = "/kaggle/input/your-tank-dataset"   # contains train/ and valid/
EPOCHS, BATCH = 40, 8
# --------------------

OUT = "/kaggle/working/rfdetr_out"                # training output
EXPORT = "/kaggle/working/export"                 # exported models
CKPT = f"{OUT}/checkpoint_best_total.pth"         # best checkpoint (name checked in section 3)
os.makedirs(EXPORT, exist_ok=True)

## 2. Train
RF-DETR Nano is Apache-2.0 and starts from COCO weights. Metrics (COCO mAP) are printed during training.

If training fails because the dataset folder is read-only, copy it first: `shutil.copytree(DATASET_DIR, "/kaggle/working/dataset")` and point `DATASET_DIR` there.

In [ ]:
from rfdetr import RFDETRNano

model = RFDETRNano()
model.train(dataset_dir=DATASET_DIR, epochs=EPOCHS, batch_size=BATCH, lr=1e-4, output_dir=OUT)

## 3. Check the result in Python

In [ ]:
print(os.listdir(OUT))   # if the best checkpoint has a different name, fix CKPT in the config cell

In [ ]:
import supervision as sv
from PIL import Image
from rfdetr import RFDETRNano

model = RFDETRNano(pretrain_weights=CKPT)

img_path = sorted(glob.glob(f"{DATASET_DIR}/valid/*.[jp][pn]g"))[0]
image = Image.open(img_path).convert("RGB")
detections = model.predict(image, threshold=0.5)

labels = [str(n) for n in detections.data["class_name"]]
annotated = sv.BoxAnnotator().annotate(image.copy(), detections)
annotated = sv.LabelAnnotator().annotate(annotated, detections, labels)
annotated

## 4. Export ONNX (browser)
The fp16 copy keeps `GridSample` in fp32, because onnxruntime-web's fp16 `GridSample` on WebGPU is reported to return wrong boxes without any error. Compare fp16 and fp32 detections in the browser before trusting the fp16 file.

Send me the printed input and output names and shapes; the browser page needs them.

In [ ]:
import onnx, onnxruntime as ort
from onnxconverter_common import float16
from rfdetr import RFDETRNano

model = RFDETRNano(pretrain_weights=CKPT)
model.export(format="onnx", output_dir=EXPORT)

onnx_path = glob.glob(f"{EXPORT}/*.onnx")[0]
s = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
print("input: ", [(i.name, i.shape) for i in s.get_inputs()])
print("output:", [(o.name, o.shape) for o in s.get_outputs()])

m16 = float16.convert_float_to_float16(
    onnx.load(onnx_path), keep_io_types=True,
    op_block_list=float16.DEFAULT_OP_BLOCK_LIST + ["GridSample"])
onnx.save(m16, f"{EXPORT}/model_fp16.onnx")

for f in sorted(glob.glob(f"{EXPORT}/*.onnx")):
    print(os.path.basename(f), round(os.path.getsize(f) / 1e6, 1), "MB")

## 5. Export LiteRT (.tflite)
Experimental in `rfdetr`: the file has a fixed input shape and runs on the CPU (XNNPACK) delegate, not the GPU. A GPU/WebGPU version needs the community two-graph conversion (litert-samples `object_detection`). Test the `.tflite` with Google's LiteRT.js model tester before building on it.

Installing the extra may change the torch version. If pip says so, restart the session (Run → Restart), run the config cell, then run only the last cell of this section.

In [ ]:
!pip install -q --no-warn-conflicts "rfdetr[litert]"

In [ ]:
from rfdetr import RFDETRNano

model = RFDETRNano(pretrain_weights=CKPT)
model.export(format="litert", output_dir=EXPORT)
print(glob.glob(f"{EXPORT}/*.tflite"))

## 6. Package

In [ ]:
shutil.copy(CKPT, EXPORT)
shutil.make_archive("/kaggle/working/export", "zip", EXPORT)
print(os.listdir(EXPORT))